In [1]:
%pip install pandas matplotlib seaborn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [37]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from scipy import stats



In [ ]:
#agregar dataset de contratos
df_contratos = pd.read_csv(r'C:\Users\Gaming\Desktop\data science\TareaData-science\data\2011\contratos.csv', index_col=False)
#agregar dataset de licitaciones
df = pd.read_csv(r'C:\Users\Gaming\Desktop\data science\TareaData-science\data\2011\records.csv', index_col=False)
#cada licitacion se puede dividir en lotes
df_lots = pd.read_csv(r'C:\Users\Gaming\Desktop\data science\TareaData-science\data\2011\ten_lots.csv', index_col=False)
#para saber en que se gasto/construye
df_items = pd.read_csv(r'C:\Users\Gaming\Desktop\data science\TareaData-science\data\2011\ten_items.csv', index_col=False)


In [9]:
variables_utiles_contratos = ['buyer.name','contracts.dateSigned','contracts.period.durationInDays','contracts.value.amount','contracts.value.currency','ocid']
df_contratos = df_contratos[variables_utiles_contratos]

variables_utiles_lots = ['compiledRelease/tender/id', 'compiledRelease/tender/lots/0/id', 'compiledRelease/tender/lots/0/title','compiledRelease/tender/lots/0/value/amount', 'compiledRelease/tender/lots/0/value/currency']
df_lots = df_lots[variables_utiles_lots]

variables_utiles_items = ['compiledRelease/tender/id', 'compiledRelease/tender/items/0/id', 'compiledRelease/tender/items/0/description', 'compiledRelease/tender/items/0/quantity', 'compiledRelease/tender/items/0/unit/name', 'compiledRelease/tender/items/0/unit/value/amount', 'compiledRelease/tender/items/0/unit/value/currency','compiledRelease/tender/items/0/relatedLot']
df_items = df_items[variables_utiles_items]

In [29]:
# 1. Renombrar df_contratos
df_contratos = df_contratos.rename(columns={
    'buyer.name': 'nombre_comprador',             
    'contracts.dateSigned': 'fecha_firma_contrato',
    'contracts.period.durationInDays': 'duracion_contrato_dias',  
    'contracts.value.amount': 'monto_contrato',
    'contracts.value.currency': 'moneda_contrato',
    'ocid': 'ocid'                               
})

# 2. Renombrar df_lots (Lotes)
df_lots = df_lots.rename(columns={
    'compiledRelease/tender/id': 'id_licitacion',   
    'compiledRelease/tender/lots/0/id': 'id_lote',
    'compiledRelease/tender/lots/0/title': 'titulo_lote',
    'compiledRelease/tender/lots/0/value/amount': 'monto_lote',
    'compiledRelease/tender/lots/0/value/currency': 'moneda_lote'
})

# 3. Renombrar df_items (Ítems)
df_items = df_items.rename(columns={
    'compiledRelease/tender/id': 'id_licitacion',  
    'compiledRelease/tender/items/0/id': 'id_item',
    'compiledRelease/tender/items/0/description': 'descripcion_item',
    'compiledRelease/tender/items/0/quantity': 'cantidad_item',
    'compiledRelease/tender/items/0/unit/name': 'unidad_medida_item',
    'compiledRelease/tender/items/0/unit/value/amount': 'monto_unitario_item',
    'compiledRelease/tender/items/0/unit/value/currency': 'moneda_item',
    'compiledRelease/tender/items/0/relatedLot': 'id_lote_relacionado'
})

In [ ]:


def datos_estadisticos(serie):
    """
    Recibe una Serie de Pandas y retorna sus estadísticas resumen.
    """
    serie_limpia = serie.dropna()
    return pd.Series({
        'Nombre Columna': serie.name,
        'Tipo de Dato': serie.dtype,
        'Valores Nulos': serie.isnull().sum(),
        'Valores Únicos': serie.nunique(),
        'Valor Mínimo': serie.min(),
        'Valor Máximo': serie.max(),
        'Media': serie.mean(),
        'Mediana': serie.median(),
        'Desviación Estándar': serie.std(),
        'Cuartiles': np.quantile(serie_limpia, [0.05,0.25, 0.5, 0.75,0.95])
    })

# Llamada a cada columna desde su DataFrame correspondiente
columnas_a_analizar = [
    df_contratos['duracion_dias'],
    df_contratos['monto_contrato'],
    df_lots['monto_lote'],
    df_items['precio_unitario'],
    df_items['cantidad']
]

# Recorremos y mostramos las estadísticas de cada una
for col in columnas_a_analizar:
    print(datos_estadisticos(col))
    print("-" * 40)

Nombre Columna                             duracion_dias
Tipo de Dato                                     float64
Valores Nulos                                        501
Valores Únicos                                       220
Valor Mínimo                                         0.0
Valor Máximo                                     11315.0
Media                                         203.168738
Mediana                                            180.0
Desviación Estándar                           285.360048
Cuartiles              [15.0, 60.0, 180.0, 270.0, 540.0]
dtype: object
----------------------------------------
Nombre Columna                                            monto_contrato
Tipo de Dato                                                       int64
Valores Nulos                                                          0
Valores Únicos                                                      7069
Valor Mínimo                                                        6768
Valor Máxi

# Medidas de distribucion o de forma

In [38]:
def forma_distribucion(serie):
    """
    Recibe una Serie de Pandas y retorna la asimetría y curtosis.
    """
    serie_limpia = serie.dropna()
    asimetria = stats.skew(serie_limpia, bias=False)
    curtosis = stats.kurtosis(serie_limpia, bias=False)
    
    return pd.Series({
        'Nombre Columna': serie.name,
        'Asimetría': asimetria,
        'Curtosis': curtosis
    })


for i in columnas_a_analizar:
    print(f'La asimetria es:{forma_distribucion(i)["Asimetría"]} y la curtosis es: {forma_distribucion(i)["Curtosis"]}')


La asimetria es:24.137646367458963 y la curtosis es: 885.0273673106456
La asimetria es:27.72694204735616 y la curtosis es: 975.724529486627
La asimetria es:227.83535289053006 y la curtosis es: 55901.36980593291
La asimetria es:425.9970791427969 y la curtosis es: 184740.54938062522
La asimetria es:100.4739375757618 y la curtosis es: 13890.406224486747
